# BioPortal Bulk Annotator

Resolve free-text strings (phenotype descriptions, taxon names, etc.) to ontology **CURIEs**
(e.g. `HP:0000252`, `NCBITAXON:562`) in bulk using the [BioPortal Annotator REST API](https://data.bioontology.org/documentation).

**Before running:**
1. Get a free API key at https://bioportal.bioontology.org/account
2. Paste it into the `API_KEY` cell below (or set the `BIOPORTAL_API_KEY` environment variable)
3. Edit the `INPUT_ITEMS` list with your own text + ontology scope
4. Run all cells

Results are collected into a pandas DataFrame and also written to `annotations.csv` / `annotations.json`.


## 1. Setup

In [ ]:
import os
import re
import csv
import json
import time
import sys
from dataclasses import dataclass, field
from typing import Optional

import requests
import pandas as pd

ANNOTATOR_URL = "https://data.bioontology.org/annotator"

# BioPortal rate-limits API keys to ~15 requests/second; stay comfortably under it.
MAX_REQUESTS_PER_SECOND = 10
MIN_INTERVAL = 1.0 / MAX_REQUESTS_PER_SECOND

MAX_RETRIES = 5
INITIAL_BACKOFF = 2.0  # seconds


## 2. API key

Set it here directly, or leave as-is to pick up the `BIOPORTAL_API_KEY` environment variable.

In [ ]:
API_KEY = os.environ.get("BIOPORTAL_API_KEY", "")

if not API_KEY or API_KEY == "":
    raise ValueError("Set API_KEY above or export BIOPORTAL_API_KEY before running.")


## 3. Helper functions

In [12]:
@dataclass
class AnnotateItem:
    """One thing you want resolved to a CURIE."""
    text: str
    ontologies: list = field(default_factory=list)   # e.g. ["HP"] or ["NCBITAXON"]
    label: Optional[str] = None                       # optional tag, e.g. "phenotype" / "taxon"
    extra_params: dict = field(default_factory=dict)  # e.g. {"longest_only": "true"}


def uri_to_curie(uri: str) -> str:
    """
    Convert a BioPortal/OBO-style class URI into a CURIE.

    http://purl.obolibrary.org/obo/HP_0000252          -> HP:0000252
    http://purl.bioontology.org/ontology/NCBITAXON/562 -> NCBITAXON:562
    Falls back to returning the original URI if no pattern matches.
    """
    m = re.search(r'/obo/([A-Za-z0-9]+)_([A-Za-z0-9]+)$', uri)
    if m:
        prefix, local_id = m.groups()
        return f"{prefix.upper()}:{local_id}"

    m = re.search(r'/ontology/([A-Za-z0-9]+)/([A-Za-z0-9]+)$', uri)
    if m:
        prefix, local_id = m.groups()
        return f"{prefix.upper()}:{local_id}"

    parts = uri.rstrip('/').split('/')
    if len(parts) >= 2:
        return f"{parts[-2].upper()}:{parts[-1]}"

    return uri


In [13]:
def _call_annotator(text: str, api_key: str, ontologies=None, extra_params=None) -> list:
    """Single throttled+retried call to the Annotator endpoint. Returns raw JSON list."""
    params = {"text": text, "apikey": api_key}
    if ontologies:
        params["ontologies"] = ",".join(ontologies)
    if extra_params:
        params.update(extra_params)

    backoff = INITIAL_BACKOFF
    for attempt in range(1, MAX_RETRIES + 1):
        resp = requests.get(ANNOTATOR_URL, params=params, timeout=30)

        if resp.status_code == 200:
            return resp.json()

        if resp.status_code == 429:
            time.sleep(backoff)
            backoff *= 2
            continue

        if resp.status_code == 401:
            raise RuntimeError("401 Unauthorized - check your API_KEY")

        sys.stderr.write(
            f"[warn] attempt {attempt}: HTTP {resp.status_code} for text={text!r}: "
            f"{resp.text[:200]}\n"
        )
        time.sleep(backoff)
        backoff *= 2

    sys.stderr.write(f"[error] giving up on text={text!r} after {MAX_RETRIES} attempts\n")
    return []


def bulk_annotate(items: list, api_key: str) -> list:
    """
    Run the Annotator over a list of AnnotateItem and return a flat list of
    result dicts, one per (input item, matched annotation) pair.
    """
    results = []
    last_call_time = 0.0

    for item in items:
        elapsed = time.time() - last_call_time
        if elapsed < MIN_INTERVAL:
            time.sleep(MIN_INTERVAL - elapsed)

        raw = _call_annotator(
            item.text,
            api_key,
            ontologies=item.ontologies,
            extra_params=item.extra_params,
        )
        last_call_time = time.time()

        if not raw:
            results.append({
                "input_text": item.text,
                "label": item.label,
                "matched_text": None,
                "curie": None,
                "uri": None,
                "ontology": None,
                "pref_label": None,
            })
            continue

        for annotation in raw:
            annotated_class = annotation.get("annotatedClass", {})
            uri = annotated_class.get("@id", "")
            curie = uri_to_curie(uri) if uri else None

            ontology_link = annotated_class.get("links", {}).get("ontology", "")
            ontology_acronym = ontology_link.rstrip("/").split("/")[-1] if ontology_link else None

            pref_label = annotated_class.get("prefLabel")

            annotations_info = annotation.get("annotations", [])
            matched_text = annotations_info[0]["text"] if annotations_info else None

            results.append({
                "input_text": item.text,
                "label": item.label,
                "matched_text": matched_text,
                "curie": curie,
                "uri": uri,
                "ontology": ontology_acronym,
                "pref_label": pref_label,
            })

    return results


## 4. Your input texts

Scope each item to the right ontology: `HP` for human phenotypes, `NCBITAXON` for taxa,
`MP` for mouse phenotypes, etc. Scoping speeds up the call and avoids false-positive
matches from unrelated ontologies.

In [22]:
INPUT_ITEMS = [
    AnnotateItem(text="microcephaly", ontologies=["HP"], label="phenotype"),
    AnnotateItem(text="cleft palate", ontologies=["HP"], label="phenotype"),
    AnnotateItem(text="short stature", ontologies=["HP"], label="phenotype"),
    AnnotateItem(text="Escherichia coli", ontologies=["NCBITAXON"], label="taxon"),
    AnnotateItem(text="Homo sapiens", ontologies=["NCBITAXON"], label="taxon"),
    AnnotateItem(text="Mus musculus", ontologies=["NCBITAXON"], label="taxon"),
    # Body sites — UBERON (Uber Anatomy Ontology)
    AnnotateItem(text="colon", ontologies=["UBERON"], label="body_site"),
    AnnotateItem(text="small intestine", ontologies=["UBERON"], label="body_site"),
    AnnotateItem(text="cecum", ontologies=["UBERON"], label="body_site"),
    AnnotateItem(text="rectum", ontologies=["UBERON"], label="body_site"),
    AnnotateItem(text="oral cavity", ontologies=["UBERON"], label="body_site"),
    AnnotateItem(text="skin", ontologies=["UBERON"], label="body_site"),
    # Diseases — MONDO (Monarch Disease Ontology)
    AnnotateItem(text="Crohn's disease", ontologies=["MONDO"], label="disease"),
    AnnotateItem(text="ulcerative colitis", ontologies=["MONDO"], label="disease"),
    AnnotateItem(text="colorectal cancer", ontologies=["MONDO"], label="disease"),
    AnnotateItem(text="type 2 diabetes", ontologies=["MONDO"], label="disease"),
    AnnotateItem(text="irritable bowel syndrome", ontologies=["MONDO"], label="disease"),
]

len(INPUT_ITEMS)

17

## 5. Run the bulk annotation

In [23]:
results = bulk_annotate(INPUT_ITEMS, API_KEY)
df = pd.DataFrame(results)
df


,input_text,label,matched_text,curie,uri,ontology,pref_label
0,microcephaly,phenotype,MICROCEPHALY,HP:0000252,http://purl.obolibrary.org/obo/HP_0000252,HP,None
1,cleft palate,phenotype,CLEFT PALATE,HP:0000175,http://purl.obolibrary.org/obo/HP_0000175,HP,None
2,short stature,phenotype,SHORT STATURE,HP:0004322,http://purl.obolibrary.org/obo/HP_0004322,HP,None
3,Escherichia coli,taxon,ESCHERICHIA COLI,NCBITAXON:562,http://purl.bioontology.org/ontology/NCBITAXON...,NCBITAXON,None
4,Escherichia coli,taxon,ESCHERICHIA,NCBITAXON:561,http://purl.bioontology.org/ontology/NCBITAXON...,NCBITAXON,None
5,Homo sapiens,taxon,HOMO SAPIENS,NCBITAXON:9606,http://purl.bioontology.org/ontology/NCBITAXON...,NCBITAXON,None
6,Homo sapiens,taxon,HOMO,NCBITAXON:9605,http://purl.bioontology.org/ontology/NCBITAXON...,NCBITAXON,None
7,Mus musculus,taxon,MUS MUSCULUS,NCBITAXON:10090,http://purl.bioontology.org/ontology/NCBITAXON...,NCBITAXON,None
8,Mus musculus,taxon,MUS,NCBITAXON:862507,http://purl.bioontology.org/ontology/NCBITAXON...,NCBITAXON,None
9,Mus musculus,taxon,MUS,NCBITAXON:10088,http://purl.bioontology.org/ontology/NCBITAXON...,NCBITAXON,None


## 6. Quick summary + save outputs

In [ ]:
matched = df["curie"].notna().sum()
print(f"{matched}/{len(df)} rows resolved to a CURIE")

df.to_csv("annotations.csv", index=False)
with open("annotations.json", "w") as f:
    json.dump(results, f, indent=2)

print("Wrote annotations.csv and annotations.json")


## 7. (Optional) Load your own texts from a CSV

If you have a CSV with columns like `text,ontology,label`, load it like this instead of
hand-typing `INPUT_ITEMS` above:

```python
src = pd.read_csv("my_texts.csv")
INPUT_ITEMS = [
    AnnotateItem(text=row.text, ontologies=[row.ontology], label=row.get("label"))
    for row in src.itertuples()
]
```